# Setlist Generator Playground

Use this notebook to experiment with the historical setlist generator.

In [16]:
from datetime import date
from pathlib import Path
import os
import sys

# Ensure repo src/ is on sys.path when running from notebooks/
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pyproject.toml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from phish_setlist_maker.db import session_scope
from phish_setlist_maker.generator import SetlistGenerator
from phish_setlist_maker.models import Show

# Confirm database connectivity
with session_scope() as session:
    latest_show = session.query(Show).order_by(Show.date.desc()).first()
    latest = latest_show.date if latest_show else 'N/A'
    print(f'Latest show in database: {latest}')


Latest show in database: 2024-10-27


In [17]:
# Configure generator parameters
reference_date = None  # e.g., date(2018, 8, 1) to reflect that tour
era = '3.0'            # None for full history or use '1.0', '2.0', '3.0', '4.0'
year = 2018            # Restrict to shows up through this calendar year
num_sets = 2           # Typical Phish show: 2 or 3
include_encore = True  # Encore block
set_lengths = {        # Override songs per segment if desired
    'set1': 10,
    'set2': 9,
    'encore': 2,
}


In [18]:
# Generate a sample setlist using the parameters above
with session_scope() as session:
    generator = SetlistGenerator(session)
    generated = generator.generate(
        reference_date=reference_date,
        era=era,
        year=year,
        num_sets=num_sets,
        include_encore=include_encore,
        set_lengths=set_lengths,
    )

generated


GeneratedSetlist(sets=[SetSegment(label='Set 1', songs=["Mike's Song", 'Stealing Time From the Faulty Plan', 'Foam', 'Funky Bitch', "Halley's Comet", 'Kill Devil Falls', 'Punch You in the Eye', 'Simple', 'McGrupp and the Watchful Hosemasters', 'Cities']), SetSegment(label='Set 2', songs=['Farmhouse', 'Theme From the Bottom', 'Twist', 'Character Zero', 'Plasma', 'Ghost', 'Carini', 'Loving Cup', 'Reba'])], encore=SetSegment(label='Encore', songs=['Sleeping Monkey', 'Tweezer Reprise']), metadata=GenerationMetadata(reference_date=datetime.date(2024, 10, 27), cutoff_date=datetime.date(2018, 12, 31), era='3.0', year=2018, notes=['Excluded 19 songs played on 2024-10-26']))

In [19]:
# Pretty-print the generated setlist
def print_segment(segment):
    print(segment.label)
    for idx, song in enumerate(segment.songs, start=1):
        print(f'  {idx}. {song}')
    print()

for segment in generated.sets:
    print_segment(segment)

if generated.encore:
    print_segment(generated.encore)

print('Metadata:')
print(f'  Reference date: {generated.metadata.reference_date}')
print(f'  Cutoff date   : {generated.metadata.cutoff_date}')
print(f'  Era           : {generated.metadata.era}')
print(f'  Year limit    : {generated.metadata.year}')

if generated.metadata.notes:
    print('Notes:')
    for note in generated.metadata.notes:
        print(f'  - {note}')
else:
    print('Notes: none')


Set 1
  1. Mike's Song
  2. Stealing Time From the Faulty Plan
  3. Foam
  4. Funky Bitch
  5. Halley's Comet
  6. Kill Devil Falls
  7. Punch You in the Eye
  8. Simple
  9. McGrupp and the Watchful Hosemasters
  10. Cities

Set 2
  1. Farmhouse
  2. Theme From the Bottom
  3. Twist
  4. Character Zero
  5. Plasma
  6. Ghost
  7. Carini
  8. Loving Cup
  9. Reba

Encore
  1. Sleeping Monkey
  2. Tweezer Reprise

Metadata:
  Reference date: 2024-10-27
  Cutoff date   : 2018-12-31
  Era           : 3.0
  Year limit    : 2018
Notes:
  - Excluded 19 songs played on 2024-10-26


In [20]:
# Explore historical encore patterns for the selected filters
from phish_setlist_maker.generator import encore_statistics

cutoff = generated.metadata.cutoff_date if 'generated' in locals() else reference_date
era_filter = generated.metadata.era if 'generated' in locals() else era
year_limit = generated.metadata.year if 'generated' in locals() else year

with session_scope() as session:
    encore_stats = encore_statistics(
        session,
        cutoff_date=cutoff,
        era=era_filter,
        year=year_limit,
    )

print('Encore count histogram (number of songs -> frequency):')
for count, freq in sorted(encore_stats.count_histogram.items()):
    avg_duration = encore_stats.average_durations_by_count.get(count, 0.0)
    print(f'  {count} songs: {freq} shows, average duration {avg_duration / 60:.1f} min')

print()
print('Top encore sequences:')
for sequence, freq in encore_stats.top_sequences[:10]:
    joined = ' → '.join(sequence)
    print(f'  {joined} ({freq} times)')

if encore_stats.longform_songs:
    print()
    print('Longform encore songs (average duration >= 12 min):')
    for title, avg_seconds in encore_stats.longform_songs:
        print(f'  {title}: {avg_seconds / 60:.1f} min')
else:
    print()
    print('No encore songs averaging at least 12 minutes in this slice.')


Encore count histogram (number of songs -> frequency):
  1 songs: 190 shows, average duration 7720.4 min
  2 songs: 152 shows, average duration 11903.8 min
  3 songs: 39 shows, average duration 15362.7 min
  4 songs: 11 shows, average duration 19333.8 min
  5 songs: 2 shows, average duration 30562.6 min
  9 songs: 1 shows, average duration 33969.2 min

Top encore sequences:
  Loving Cup (25 times)
  Character Zero (22 times)
  Sleeping Monkey → Tweezer Reprise (14 times)
  Good Times Bad Times (13 times)
  Julius (11 times)
  A Day in the Life (10 times)
  Loving Cup → Tweezer Reprise (10 times)
  Shine a Light (9 times)
  First Tube (9 times)
  Quinn the Eskimo (The Mighty Quinn) (8 times)

Longform encore songs (average duration >= 12 min):
  You Enjoy Myself: 20648.6 min
  Fluffhead: 15609.2 min
  Reba: 14753.7 min
  Harry Hood: 13340.7 min
  Bathtub Gin: 12125.6 min
  Sneakin' Sally Through the Alley: 11911.4 min
  Walls of the Cave: 10870.6 min
  Run Like an Antelope: 10727.5 min
